# 1. Download data

### [Quran Tafseer RAG Dataset](https://huggingface.co/datasets/omaressam1111/multi-tafseer-quran-rag)
A structured Arabic dataset of Quranic tafseer collected from eight classical and modern tafseer books.
The dataset contains verse-aligned tafseer passages designed for Retrieval-Augmented Generation (RAG) systems and Arabic NLP research.

Each record links a Quran verse with its corresponding tafseer explanation from one of the tafseer books and includes rich metadata such as surah information, tafseer source, and embedding-ready text.


In [1]:
from datasets import load_dataset

ds = load_dataset("omaressam1111/multi-tafseer-quran-rag")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

master_tafseer_dataset.csv:   0%|          | 0.00/110M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/49887 [00:00<?, ? examples/s]

In [2]:
ds

DatasetDict({
    train: Dataset({
        features: ['chunk_id', 'book_api_id', 'book_slug', 'book_name_ar', 'book_name_en', 'author', 'surah_number', 'surah_name_ar', 'surah_name_en', 'revelation_type', 'ayah_number_start', 'ayah_number_end', 'juz', 'ayah_text', 'tafseer_text', 'text_for_embedding', 'word_count', 'char_count', 'is_self_referential'],
        num_rows: 49887
    })
})

In [3]:
docs = [i['text_for_embedding'] for i in ds['train'].select(range(10000))]
len(docs)

10000

In [4]:
import numpy as np
np.max([len(doc) for doc in docs])

np.int64(23497)

In [5]:
docs[-1]

'passage: سورة يس - الآية 59\nالآية: وَٱمْتَٰزُوا۟ ٱلْيَوْمَ أَيُّهَا ٱلْمُجْرِمُونَ\nالتفسير (تفسير الجلالين): «و» يقول «امتازوا اليوم أيها المجرمون» أي انفردوا عن المؤمنين عند اختلاطهم بهم.'

In [6]:
docs[:10]

['passage: سورة الفاتحة - الآية 1\nالآية: \ufeffبِسْمِ ٱللَّهِ ٱلرَّحْمَٰنِ ٱلرَّحِيمِ\nالتفسير (تفسير البغوي): ( بسم الله الرحمن الرحيم ) بسم الله الباء أداة تخفض ما بعدها مثل من وعن والمتعلق به الباء محذوف لدلالة الكلام عليه تقديره أبدأ بسم الله أو قل بسم الله . وأسقطت الألف من الاسم طلبا للخفة وكثرة استعمالها وطولت الباء قال القتيبي ليكون افتتاح كلام كتاب الله بحرف معظم كان عمر بن عبد العزيز رحمه الله يقول لكتابه طولوا الباء وأظهروا السين وفرجوا بينهما ودوروا الميم . تعظيما لكتاب الله تعالى وقيل لما أسقطوا الألف ردوا طول الألف على الباء ليكون دالا على سقوط الألف ألا ترى أنه لما كتبت الألف في " اقرأ باسم ربك " ( 1 - العلق ) ردت الباء إلى صيغتها ولا تحذف الألف إذا أضيف الاسم إلى غير الله ولا مع غير الباء .والاسم هو المسمى وعينه وذاته قال الله تعالى : " إنا نبشرك بغلام اسمه يحيى " ( 7 - مريم ) أخبر أن اسمه يحيى ثم نادى الاسم فقال : يا يحيى " وقال : ما تعبدون من دونه إلا أسماء سميتموها " ( 40 - يوسف ) وأراد الأشخاص المعبودة لأنهم كانوا يعبدون المسميات وقال : سبح اسم ربك " ( 1 - الأعلى )

# 2. Chunking

Resources:
1. [Text Splitting in LangChain: A Deep Dive into Efficient Chunking Methods](https://medium.com/@anixlynch/7-chunking-strategies-for-langchain-b50dac194813#c754)
2. [7 Chunking Strategies for Langchain📖](https://medium.com/@anixlynch/7-chunking-strategies-for-langchain-b50dac194813#c754)
3. [Splitting by token - Text splitter integration guide](https://docs.langchain.com/oss/python/integrations/splitters/split_by_token)
4. [Mastering Text Splitting in Langchain](https://medium.com/@divakar1591/mastering-text-splitting-in-langchain-4750716e4090)
5. [Tiktoken Encoding Models](https://github.com/openai/tiktoken/blob/main/tiktoken/model.py)

In [7]:
!pip install --upgrade -q langchain-text-splitters tiktoken

In [8]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter, TokenTextSplitter

In [9]:
# 1. fixed chunking
def fixed_chunking(text):

    splitter = CharacterTextSplitter(
        separator="",
        chunk_size=100,
        chunk_overlap=30,
    )

    return splitter.split_text(text)

fixed_chunks = []

for doc in docs[:1000]:
    # print(doc)
    fixed_chunk = fixed_chunking(doc)

    fixed_chunks.extend(fixed_chunk)

print("Fixed Chunking")
print("Number of chunks:", len(fixed_chunks))
print("=" * 60)

for i, chunk in enumerate(fixed_chunks):
    print(f"\nChunk {i+1}:")
    print(len(chunk),chunk)

Streaming output truncated to the last 5000 lines.

Chunk 21825:
99 يعرفها ولم يصدقها فعليها ، أي : فبنفسه ضر ، ووبال العمى عليه ، ( وما أنا عليكم بحفيظ ) برقيب أحصي ع

Chunk 21826:
99 أنا عليكم بحفيظ ) برقيب أحصي عليكم أعمالكم ، إنما أنا رسول إليكم أبلغكم رسالات ربي وهو الحفيظ عليكم

Chunk 21827:
64 م رسالات ربي وهو الحفيظ عليكم الذي لا يخفى عليه شيء من أفعالكم .

Chunk 21828:
100 passage: سورة الأنعام - الآية 105
الآية: وَكَذَٰلِكَ نُصَرِّفُ ٱلْءَايَٰتِ وَلِيَقُولُوا۟ دَرَسْتَ و

Chunk 21829:
99 َٰتِ وَلِيَقُولُوا۟ دَرَسْتَ وَلِنُبَيِّنَهُۥ لِقَوْمٍۢ يَعْلَمُونَ
التفسير (تفسير البغوي): ( وكذلك

Chunk 21830:
100 تفسير (تفسير البغوي): ( وكذلك نصرف الآيات ) نفصلها ونبينها في كل وجه ، ( وليقولوا ) قيل : معناه لئلا

Chunk 21831:
99 ( وليقولوا ) قيل : معناه لئلا يقولوا ، ( درست ) وقيل : هذه اللام لام العاقبة أي عاقبة أمرهم أن يقول

Chunk 21832:
100 العاقبة أي عاقبة أمرهم أن يقولوا : درست ، أي : قرأت على غيرك ، وقيل : قرأت كتب أهل الكتاب ، كقوله تع

Chunk 21833:
100 قرأت كتب أهل الكتاب ، كق

In [10]:
# 2. recursive chunking
def recursive_chunking(text):

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=100,
        chunk_overlap=30,
        separators=["\n\n", "\n", ".", " "]
    )

    return splitter.split_text(text)

recursive_chunks = []

for doc in docs[:1000]:

    recursive_chunk = recursive_chunking(doc)

    recursive_chunks.extend(recursive_chunk)

print("Recursive Chunking")
print("Number of chunks:", len(recursive_chunks))
print("=" * 60)

for i, chunk in enumerate(recursive_chunks):
    print(f"\nChunk {i+1}:")
    print(len(chunk), chunk)

Streaming output truncated to the last 5000 lines.
Chunk 23219:
98 عليه السلام كان يحيي الموتى فأتنا من الآيات حتى نصدقك ، فقال رسول الله صلى الله عليه وسلم : أي شيء

Chunk 23220:
96 صلى الله عليه وسلم : أي شيء تحبون؟ قالوا : تجعل لنا الصفا ذهبا أو ابعث لنا بعض أمواتنا حتى نسأله

Chunk 23221:
99 لنا بعض أمواتنا حتى نسأله عنك أحق ما تقول أم باطل ، أو أرنا الملائكة يشهدون لك ، فقال رسول الله صلى

Chunk 23222:
96 لك ، فقال رسول الله صلى الله عليه وسلم : فإن فعلت بعض ما تقولون أتصدقونني؟ قالوا : نعم والله لئن

Chunk 23223:
96 قالوا : نعم والله لئن فعلت لنتبعنك أجمعين ، وسأل المسلمون رسول الله صلى الله عليه وسلم أن ينزلها

Chunk 23224:
99 صلى الله عليه وسلم أن ينزلها عليهم حتى يؤمنوا ، فقام رسول الله صلى الله عليه وسلم يدعو الله أن يجعل

Chunk 23225:
98 عليه وسلم يدعو الله أن يجعل الصفا ذهبا فجاءه جبريل عليه السلام ، فقال له : اختر ما شئت إن شئت أصبح

Chunk 23226:
98 له : اختر ما شئت إن شئت أصبح ذهبا ولكن إن لم يصدقوا عذبتهم ، وإن شئت تركتهم حتى يتوب تائبهم ، فقال

Chunk 23227:
99 تركتهم حت

In [11]:
# 3. token splitter
def token_splitter(text):

    splitter = TokenTextSplitter(
        encoding_name = 'cl100k_base',
        chunk_size=100,
        chunk_overlap=0
      )

    return splitter.split_text(text)

token_chunks = []

for doc in docs[:1000]:

    token_chunk = token_splitter(doc)

    token_chunks.extend(token_chunk)

print("Token Splitter")
print("Number of chunks:", len(token_chunks))
print("=" * 60)

for i, chunk in enumerate(token_chunks):
    print(f"\nChunk {i+1}:")
    print(len(chunk), chunk)

Streaming output truncated to the last 5000 lines.
Chunk 10995:
136 كم ) على ما أقول ، ويشهد لي بالحق وعليكم بالباطل ، ( وأوحي إلي هذا القرآن لأنذركم به ) لأخوفكم به يا أهل مكة ، ( ومن بلغ ) يعني : ومن بل

Chunk 10996:
141 غه القرآن من العجم وغيرهم من الأمم إلى يوم القيامة .حدثنا أبو الفضل زياد بن محمد بن الحنفي أنا محمد بن بشر بن محمد المزني أنا أبو بكر محمد بن

Chunk 10997:
143  الحسن بن بشر النقاش أنا أبو شعيب الحراني أنا يحيى بن عبد الله بن الضحاك البابلي أنا الأوزاعي حدثني حسان بن عطية عن أبي كبشة [ السلولي ] عن عبد

Chunk 10998:
145  الله بن عمرو قال : قال رسول الله صلى الله عليه وسلم : " بلغوا عني ولو آية ، وحدثوا عن بني إسرائيل ولا حرج ومن كذب علي متعمدا فليتبوأ مقعده من ال

Chunk 10999:
144 نار " .أخبرنا أبو الحسن عبد الوهاب بن محمد الخطيب أخبرنا عبد العزيز بن أحمد الخلال أنا أبو العباس الأصم أنا الربيع أنا الشافعي أنا سفيان بن عيين

Chunk 11000:
146 ة عن عبد الملك بن عمير عن عبد الرحمن بن عبد الله بن مسعود عن أبيه أن رسول الله صلى الله عليه وسلم قال : " نضر الله عبدا سمع مقال

In [12]:
print(f"len of fixed chunks: {len(fixed_chunks)}")
print(f"len of recursive chunks: {len(recursive_chunks)}")
print(f"len of token chunks: {len(token_chunks)}")


len of fixed chunks: 23405
len of recursive chunks: 24883
len of token chunks: 12533


# 2. Embeddings

In [13]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("NAMAA-Space/AraModernBert-Base-STS")

# sentences = [
#     "الذكاء الاصطناعي يغير طريقة تفاعلنا مع التكنولوجيا.",
#     "التكنولوجيا تتطور بسرعة بفضل الذكاء الاصطناعي.",
#     "الذكاء الاصطناعي يسهم في تطوير التطبيقات الذكية.",
#     "تحديات الذكاء الاصطناعي تشمل الحفاظ على الأمان والأخلاقيات."
# ]
# embeddings = model.encode(sentences)

# similarities = model.similarity(embeddings, embeddings)
# print(similarities.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/205 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [14]:
# similarities

In [ ]:
from tqdm.auto import tqdm

fixed_database = [(chunk, model.encode(chunk)) for chunk in tqdm(fixed_chunks, desc="Encoding fixed chunks")]
recursive_database = [(chunk, model.encode(chunk)) for chunk in tqdm(recursive_chunks, desc="Encoding recursive chunks")]
token_database = [(chunk, model.encode(chunk)) for chunk in tqdm(token_chunks, desc="Encoding token chunks")]

len(fixed_database), len(recursive_database), len(token_database)

Encoding fixed chunks:   0%|          | 0/23405 [00:00<?, ?it/s]

Encoding recursive chunks:   0%|          | 0/24883 [00:00<?, ?it/s]

Encoding token chunks:   0%|          | 0/12533 [00:00<?, ?it/s]

(23405, 24883, 12533)

In [16]:
model.similarity_fn_name

'cosine'

In [17]:
def search(query, database, k=3):
  query_emb = model.encode(query)
  searched_vectors = [(doc, model.similarity(query_emb, emb).item()) for (doc, emb) in database]
  ordered = sorted(searched_vectors, key=lambda x: x[1], reverse=True)
  return ordered[0:k]

In [18]:
results = search("ما تفسير ذلك الكتاب لا ريب فيه", recursive_database)

In [19]:
results

[('القرآن، وقيل: هذا فيه مضمر أي هذا ذلك الكتاب', 0.6896310448646545),
 ('التفسير (تفسير البغوي): قوله تعالى: {ذلك الكتاب}: أي هذا الكتاب وهو القرآن، وقيل: هذا فيه مضمر أي',
  0.6614440083503723),
 ('وكلا الفريقين يقرءون الكتاب ، قيل : معناه ليس في كتبهم هذا الاختلاف فدل تلاوتهم الكتاب ومخالفتهم ما',
  0.6052238345146179)]

In [20]:
def get_database(method):

    if method == "Fixed Chunking":
        return fixed_database

    elif method == "Recursive Chunking":
        return recursive_database

    elif method == "Token Splitter":
        return token_database

    else:
        return recursive_database

In [21]:
def build_prompt(query, results):

    context = ""

    for i, (doc, score) in enumerate(results):

        context += f"\n[المصدر {i+1}]\n"
        context += doc
        context += "\n"

    prompt = f"""
أنت مساعد عربي متخصص في الإجابة اعتمادًا على النصوص المسترجعة فقط.

التعليمات:
- أجب باللغة العربية.
- استخدم السياق الموجود فقط.
- لا تضف معلومات من خارج السياق.
- إذا لم تجد الإجابة في السياق، قل: لا توجد معلومات كافية في النصوص المسترجعة.

السياق:
{context}

السؤال:
{query}

الإجابة:
"""

    return prompt

In [22]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

llm_name = "microsoft/Phi-4-mini-instruct"

tokenizer = AutoTokenizer.from_pretrained(llm_name)

llm_model = AutoModelForCausalLM.from_pretrained(
    llm_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

config.json: 0.00B [00:00, ?B/s]

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

In [23]:
def generate_answer(prompt):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(llm_model.device)

    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=300
    )

    answer_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    answer = tokenizer.decode(
        answer_tokens,
        skip_special_tokens=True
    )

    return answer

In [24]:
def rag_answer(query, method, k=3):

    database = get_database(method)

    results = search(
        query,
        database,
        k=k
    )

    prompt = build_prompt(
        query,
        results
    )

    answer = generate_answer(prompt)

    return answer, results

In [32]:
answer, results = rag_answer("ما تفسير ذلك الكتاب لا ريب فيه", "Fixed Chunking")

In [33]:
answer

'تفسير ذلك الكتاب لا ريب فيه هو أن الكتاب الذي ينص على "لا ريب فيه" هو القرآن الكريم، وأنه لا يوجد فيه أي شك أو اختلاف، وأنه هو كلام الله تعالى الذي نزل إليك من دون أي تأليف أو اختياري، كما ورد في المصادر المعطاة.'

In [34]:
results

[('كتابا لأنه جمع حرف إلى حرف.قوله تعالى: {لا ريب فيه}: أي لا شك فيه أنه من عند الله عز وجل وأنه الحق و',
  0.6886751055717468),
 ('قال: {ذلك الكتاب} يعني ما تقدم البقرة من السور لا شك فيه".والكتاب: مصدر وهو بمعنى المكتوب؛ كما يقال',
  0.6426488757133484),
 ('مِنِينَ\nالتفسير (تفسير البغوي): ( كتاب ) أي : هذا كتاب ، ( أنزل إليك ) وهو القرآن ، ( فلا يكن في صدر',
  0.6401673555374146)]

In [28]:
!pip -q install gradio

In [29]:
import gradio as gr

def chatbot_fn(user_question, method):

    answer, results = rag_answer(
        user_question,
        method=method,
        k=3
    )

    sources = ""

    for i, (doc, score) in enumerate(results):

        sources += f"\n\nالمصدر {i+1} | Score: {score:.4f}\n"
        sources += doc[:700]
        sources += "\n" + "-" * 60

    final_output = answer
    final_output += "\n\nالنصوص المسترجعة:"
    final_output += sources

    return final_output

In [30]:
with gr.Blocks() as demo:

    gr.Markdown("## Arabic Tafseer RAG Chatbot")

    question_box = gr.Textbox(
        label="السؤال",
        placeholder="مثال: لماذا سميت سورة الفاتحة بهذا الاسم؟"
    )

    method_box = gr.Radio(
        choices=[
            "Fixed Chunking",
            "Recursive Chunking",
            "Token Splitter"
        ],
        value="Recursive Chunking",
        label="Chunking Method"
    )

    ask_button = gr.Button("Ask")

    output_box = gr.Textbox(
        label="Answer",
        lines=20
    )

    ask_button.click(
        chatbot_fn,
        inputs=[question_box, method_box],
        outputs=output_box
    )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3aaed3d9fe38dd320a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [31]:
#use bm25 search